<a href="https://colab.research.google.com/github/10kshaodow/Deep-Learning-Lab-2/blob/main/Lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib as plt
import torch
import torch.nn.functional as f
from datasets import load_dataset
import os
import math
import random
from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import os
from datasets import load_dataset, load_from_disk

# https://huggingface.co/datasets/wmt/wmt14
ds = load_dataset("wmt/wmt14", "de-en")
train_ds = ds["train"]
test_ds = ds["test"]
val_ds = ds["validation"]

tokenizer_train_ds = train_ds.select(range(50000))

tokenizer = None
tokenizer_path = "/content/drive/MyDrive/CMPE-679-Labs/Lab2-Transformer/my_trained_tokenizer.json"

if not os.path.exists(tokenizer_path):
    print(f"Tokenizer file not found at {tokenizer_path}. Starting training...")

    tokenizer_core = Tokenizer(BPE(unk_token="[UNK]"))

    trainer = BpeTrainer(
        vocab_size=37000,
        special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
    )

    tokenizer_core.pre_tokenizer = Whitespace()

    def tokenizer_generator(ds):
        for row in ds:
            yield row["translation"]["en"]
            yield row["translation"]["de"]

    tokenizer_dir = os.path.dirname(tokenizer_path)
    os.makedirs(tokenizer_dir, exist_ok=True)

    tokenizer_core.train_from_iterator(tokenizer_generator(tokenizer_train_ds), trainer)
    tokenizer_core.save(tokenizer_path)

    print(f"Tokenizer trained and saved to {tokenizer_path}.")

    tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer_core)
    tokenizer.pad_token = "[PAD]"
    tokenizer.bos_token = "[BOS]"
    tokenizer.eos_token = "[EOS]"

else:
    print(f"Tokenizer file found at {tokenizer_path}. Loading existing tokenizer...")
    tokenizer = PreTrainedTokenizerFast(tokenizer_file=tokenizer_path)
    tokenizer.pad_token = "[PAD]"
    tokenizer.bos_token = "[BOS]"
    tokenizer.eos_token = "[EOS]"
    print("Tokenizer loaded successfully.")


max_len = 128
batch_size = 128


def preprocess(batch):
    src = batch["translation"]["de"]
    tgt = batch["translation"]["en"]

    model_inputs = tokenizer(src, max_length=max_len, truncation=True)
    labels = tokenizer(text_target=tgt, max_length=max_len, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


# Paths to save/load processed datasets
processed_base_path = "/content/drive/MyDrive/CMPE-679-Labs/Lab2-Transformer/processed_wmt14_de_en"
train_tokens_path = os.path.join(processed_base_path, "train_tokens")
test_tokens_path = os.path.join(processed_base_path, "test_tokens")
val_tokens_path = os.path.join(processed_base_path, "val_tokens")

# If already processed, load from disk
if (
    os.path.exists(train_tokens_path) and
    os.path.exists(test_tokens_path) and
    os.path.exists(val_tokens_path)
):
    print("Processed datasets found. Loading from disk...")
    train_tokens = load_from_disk(train_tokens_path)
    test_tokens = load_from_disk(test_tokens_path)
    val_tokens = load_from_disk(val_tokens_path)
    print("Processed datasets loaded successfully.")

else:
    print("Processed datasets not found. Running preprocessing...")

    train_tokens = train_ds.map(
        preprocess,
        remove_columns=train_ds.column_names,
        num_proc=4
    )

    test_tokens = test_ds.map(
        preprocess,
        remove_columns=test_ds.column_names,
        num_proc=4
    )

    val_tokens = val_ds.map(
        preprocess,
        remove_columns=val_ds.column_names,
        num_proc=4
    )

    os.makedirs(processed_base_path, exist_ok=True)

    train_tokens.save_to_disk(train_tokens_path)
    test_tokens.save_to_disk(test_tokens_path)
    val_tokens.save_to_disk(val_tokens_path)

    print("Processed datasets saved to disk.")


collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=tokenizer.pad_token_id
)

train_loader = DataLoader(
    train_tokens,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collator
)

test_loader = DataLoader(
    test_tokens,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collator
)

val_loader = DataLoader(
    val_tokens,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collator
)

Tokenizer file found at /content/drive/MyDrive/CMPE-679-Labs/Lab2-Transformer/my_trained_tokenizer.json. Loading existing tokenizer...
Tokenizer loaded successfully.
Processed datasets found. Loading from disk...
Processed datasets loaded successfully.


#Positional Encoding

In [10]:
import torch.nn as nn
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        # Create a matrix of shape (max_len, d_model) filled with zeros
        pe = torch.zeros(max_len, d_model)

        # Create a vector of positions (0, 1, 2, ..., max_len-1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # Calculate the division term for the sine and cosine functions
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # Apply sine to even indices and cosine to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add a batch dimension and register as a buffer (not a trainable parameter)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        x shape: (batch_size, seq_len, d_model)

        Adds the positional encoding vectors to
        the token embeddings so the model knows
        the order of words in the sequence.
        """

        seq_len = x.size(1) # extracts the x_shape seq_len(size of sentence)
        return x + self.pe[:, :seq_len, :]

#Scaled attention

In [11]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, Q, K, V, mask=None):
        """
        Q shape: (batch_size, num_heads, seq_len_q, d_k)
        K shape: (batch_size, num_heads, seq_len_k, d_k)
        V shape: (batch_size, num_heads, seq_len_v, d_v)
        mask shape: broadcastable to (batch_size, num_heads, seq_len_q, seq_len_k)
        """

        # Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(Q.size(-1))
        # scores shape: (batch_size, num_heads, seq_len_q, seq_len_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(scores, dim=-1)

        output = torch.matmul(attention_weights, V)
        # output shape: (batch_size, num_heads, seq_len_q, d_v)

        return output, attention_weights

#Multi-Head Attention

In [12]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # Final output projection
        self.W_o = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention()

    def split_heads(self, x):
        """
        x shape: (batch_size, seq_len, d_model)
        returns: (batch_size, num_heads, seq_len, d_k)
        """
        batch_size, seq_len, _ = x.size()

        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        x = x.transpose(1, 2)

        return x

    def combine_heads(self, x):
        """
        x shape: (batch_size, num_heads, seq_len, d_k)
        returns: (batch_size, seq_len, d_model)
        """
        batch_size, num_heads, seq_len, d_k = x.size()

        x = x.transpose(1, 2).contiguous()
        x = x.view(batch_size, seq_len, num_heads * d_k)

        return x

    def forward(self, Q, K, V, mask=None):
        """
        Q, K, V shape: (batch_size, seq_len, d_model)
        mask shape: broadcastable to (batch_size, num_heads, seq_len_q, seq_len_k)
        """

        # Linear projections
        Q = self.W_q(Q)
        K = self.W_k(K)
        V = self.W_v(V)

        # Split into heads
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        # Apply attention
        attention_output, attention_weights = self.attention(Q, K, V, mask)

        # Combine heads
        combined_output = self.combine_heads(attention_output)

        # Final linear layer
        output = self.W_o(combined_output)

        return output, attention_weights

#Feed Forward

In [13]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()

        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        """
        x shape: (batch_size, seq_len, d_model)
        """
        return self.linear2(self.dropout(self.relu(self.linear1(x))))

#Encoder Layer

In [14]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        """
        x shape: (batch_size, src_seq_len, d_model)
        src_mask shape: broadcastable to attention scores
        """

        # Self-attention
        attn_output, attn_weights = self.self_attention(x, x, x, src_mask)
        x = self.norm1(x + self.dropout1(attn_output))

        # Feed-forward
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout2(ff_output))

        return x, attn_weights

#Decoder Layer

In [15]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.cross_attention = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_output, tgt_mask=None, src_mask=None):
        """
        x shape: (batch_size, tgt_seq_len, d_model)
        enc_output shape: (batch_size, src_seq_len, d_model)
        tgt_mask: mask for decoder self-attention
        src_mask: mask for encoder-decoder attention
        """

        # Masked self-attention
        self_attn_output, self_attn_weights = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(self_attn_output))

        # Cross-attention: queries from decoder, keys/values from encoder
        cross_attn_output, cross_attn_weights = self.cross_attention(
            x, enc_output, enc_output, src_mask
        )
        x = self.norm2(x + self.dropout2(cross_attn_output))

        # Feed-forward
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout3(ff_output))

        return x, self_attn_weights, cross_attn_weights